# Stochastic Calculus, Step by Step · Lesson 2: Multivariable calculus

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **The course's numbers.** A share priced at S₀ = 100, a strike K = 105, a risk-free rate r = 3% (continuously compounded), a volatility σ = 20% and a horizon T = 1 year. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. Partial derivatives · 3. The multivariable chain rule · 4. Two-variable Taylor, with the cross term · 5. The equation dm/dt = km · 6. Exercises · 7. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 2,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

## 2. Partial derivatives

Take $g(t, x) = e^{-rt}x^2$: a value that depends on time and on a price. Each partial derivative moves one variable and freezes the other:

$$g_t = -r e^{-rt}x^2, \quad g_x = 2e^{-rt}x, \quad g_{xx} = 2e^{-rt}, \quad g_{tx} = -2re^{-rt}x, \quad g_{tt} = r^2e^{-rt}x^2 .$$

Check them by finite differences at $(t, x) = (0.5, 100)$.

In [ ]:
g = lambda t, x: math.exp(-r * t) * x**2
t0, x0 = 0.5, S0
exact = {"g": g(t0, x0), "gt": -r * math.exp(-r * t0) * x0**2, "gx": 2 * math.exp(-r * t0) * x0,
         "gxx": 2 * math.exp(-r * t0), "gtx": -2 * r * math.exp(-r * t0) * x0, "gtt": r**2 * math.exp(-r * t0) * x0**2}
h, k = 1e-4, 1e-3
fd = {"gt": (g(t0 + h, x0) - g(t0 - h, x0)) / (2 * h),
      "gx": (g(t0, x0 + k) - g(t0, x0 - k)) / (2 * k),
      "gxx": (g(t0, x0 + k) - 2 * g(t0, x0) + g(t0, x0 - k)) / k**2,
      "gtx": (g(t0 + h, x0 + k) - g(t0 + h, x0 - k) - g(t0 - h, x0 + k) + g(t0 - h, x0 - k)) / (4 * h * k)}
assert all(abs(fd[key] / exact[key] - 1) < 1e-5 for key in fd)
out["partials"] = {"t0": t0, "x0": x0, **exact, "fd": fd}
# the picture: freeze t at t0 and move x; the slope at x0 is g_x
xs = np.linspace(50, 150, 101)
out["partials"]["slice"] = {"x": xs.tolist(), "g": (np.exp(-r * t0) * xs**2).round(3).tolist(),
                            "tangent": (exact["g"] + exact["gx"] * (xs - x0)).round(3).tolist(),
                            "ticks": [{"x": v, "label": f"{v:g}"} for v in (50, 75, 100, 125, 150)]}
pd.DataFrame({"exact": {kk: exact[kk] for kk in fd}, "finite difference": fd})

## 3. The multivariable chain rule

Along a path $x(t)$, the total change of $g(t, x(t))$ collects both partials: $\frac{d}{dt}g(t, x(t)) = g_t + g_x\,x'(t)$. Along the forward path $x(t) = S_0e^{rt}$, $h(t) = g(t, x(t)) = S_0^2e^{rt}$, so $h'(t) = r\,h(t)$.

In [ ]:
x_path = lambda t: S0 * math.exp(r * t)
xt = x_path(t0)
chain = -r * math.exp(-r * t0) * xt**2 + 2 * math.exp(-r * t0) * xt * (r * xt)     # g_t + g_x * x'(t)
direct = r * S0**2 * math.exp(r * t0)
numeric = (g(t0 + 1e-5, x_path(t0 + 1e-5)) - g(t0 - 1e-5, x_path(t0 - 1e-5))) / 2e-5
assert abs(chain - direct) < 1e-9 and abs(numeric / direct - 1) < 1e-8
out["chain"] = {"chain": chain, "direct": direct, "numeric": numeric}
out["chain"]

## 4. Two-variable Taylor, with the cross term

$$\Delta g \approx g_t\Delta t + g_x\Delta x + \tfrac12\big(g_{tt}\Delta t^2 + 2g_{tx}\Delta t\,\Delta x + g_{xx}\Delta x^2\big).$$

Take one trading day, $\Delta t = 1/252$, and a typical one-day price move $\Delta x = \sigma x\sqrt{\Delta t}$. Which terms matter?

In [ ]:
dt = 1 / 252
dx = sigma * x0 * math.sqrt(dt)
actual = g(t0 + dt, x0 + dx) - g(t0, x0)
terms = {"g_t dt": exact["gt"] * dt, "g_x dx": exact["gx"] * dx, "1/2 g_tt dt^2": 0.5 * exact["gtt"] * dt**2,
         "g_tx dt dx": exact["gtx"] * dt * dx, "1/2 g_xx dx^2": 0.5 * exact["gxx"] * dx**2}
first = terms["g_t dt"] + terms["g_x dx"]
second = sum(terms.values())
out["taylor2"] = {"dt": dt, "dx": dx, "actual": actual, "first": first, "second": second,
                  "first_err": first - actual, "second_err": second - actual,
                  "rows": [{"term": kk, "value": v, "label": lab} for (kk, v), lab in zip(terms.items(), (
                      "time: g_t Δt", "price: g_x Δx", "½ g_tt Δt²", "cross: g_tx Δt Δx", "½ g_xx Δx²"))],
                  "xx_over_t": terms["1/2 g_xx dx^2"] / abs(terms["g_t dt"])}
# how big each kind of step is over one day, when dx is of size sqrt(dt)
out["orders"] = {"rows": [{"what": "Δt", "power": "1", "value": dt},
                          {"what": "Δx ≈ √Δt", "power": "1/2", "value": math.sqrt(dt)},
                          {"what": "Δt²", "power": "2", "value": dt**2},
                          {"what": "Δt Δx", "power": "3/2", "value": dt * math.sqrt(dt)},
                          {"what": "Δx²", "power": "1", "value": dt}]}
# the picture: the surface and its tangent plane at (t0, x0)
tt, xx = np.meshgrid(np.linspace(0, 1, 26), np.linspace(20, 180, 33))
out["surface"] = {"t": tt.round(4).tolist(), "x": xx.round(3).tolist(), "g": (np.exp(-r * tt) * xx**2).round(3).tolist(),
                  "plane": (exact["g"] + exact["gt"] * (tt - t0) + exact["gx"] * (xx - x0)).round(3).tolist(),
                  "t0": t0, "x0": x0, "g0": exact["g"]}
pd.DataFrame(out["taylor2"]["rows"])

## 5. The equation dm/dt = km

If a quantity grows at a rate proportional to itself, $m'(t) = k\,m(t)$, then $m(t) = m(0)e^{kt}$. Lesson 17 meets it with $k = 2r + \sigma^2$ and $m(0) = S_0^2$, where $m(t)$ will be $E[S_t^2]$. Stepping the equation forward (Euler) converges to the exact answer:

In [ ]:
kk = 2 * r + sigma**2
m_exact = S0**2 * math.exp(kk * T)
euler = []
for n in (1, 10, 100, 1000):
    m = S0**2
    for _ in range(n):
        m += kk * m * (T / n)
    euler.append({"steps": n, "euler": m, "error": m - m_exact})
out["ode"] = {"k": kk, "m0": S0**2, "exact": m_exact, "rows": euler}
pd.DataFrame(euler)

## 6. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 1: Solutions*.

**Exercise 1.** For $f(t, x) = \ln x + (r - \tfrac12\sigma^2)(T - t)$, compute $f_t$, $f_x$ and $f_{xx}$.

**Exercise 2.** Let $f(t, x) = t\,x^2$ and $x(t) = e^{at}$. Find $\frac{d}{dt}f(t, x(t))$ twice: with the chain rule, and by substituting first.

**Exercise 3.** Expand $f(x, y) = xy$ around $(1, 2)$ to second order. Show that the expansion is exact, and that dropping the cross term leaves an error of exactly $\Delta x\,\Delta y$.

**Exercise 4.** Solve $m'(t) = k\,m(t) + c$ with $m(0) = m_0$. Evaluate $m(1)$ for $k = 0.1$, $c = 5$, $m_0 = 100$.

**Exercise 5.** Find all the first and second partial derivatives of $f(t, x) = t\,x^2 + e^{tx}$, check that $f_{tx} = f_{xt}$, and evaluate $f_{tx}$ at $(t, x) = (1, 0.5)$.

**Exercise 6.** Show that $f(t, x) = e^{x - t/2}$ and $g(t, x) = x^2 - t$ both satisfy $f_t + \tfrac12f_{xx} = 0$. For which constant $c$ does $x^3 - c\,t\,x$ satisfy it too?

**Exercise 7.** (Code) For $g(t, x) = e^{-rt}x^2$ at $(0.5, 100)$, halve $(\Delta t, \Delta x)$ repeatedly and show that the second-order Taylor error falls about 8 times each time.

**Exercise 8.** (Code) Solve $m' = km$ with $k = 0.10$, $m(0) = 10{,}000$ by Euler steps, $n = 10, 100, 1000, 10{,}000$. Show that the error falls in proportion to $1/n$.

## 7. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")